# Fase 4 — Modelado Predictivo con Aprendizaje Computacional

**Proyecto:** Olist E-Commerce — Análisis del marketplace brasileño
**Materia:** Análisis de Datos — Aprendizaje Computacional · ITM
**Fecha:** Mayo 2026
**Dataset:** Olist (~96k órdenes entregadas tras filtros)
**Target:** `is_positive_review` (binaria · 1 = reseña ≥ 4 estrellas)
**Mejor modelo final:** XGBoost · **ROC-AUC test = 0.705** · **F1 = 0.837**

---

## Contenido

1. Objetivos, alcance y métricas de éxito
2. Carga de datos
3. Aplicación del plan de outliers (Fase 2 · sección 12.3)
4. Selección y construcción de features
5. Split train/test estratificado
6. Pipeline de preprocesamiento (sklearn `ColumnTransformer`)
7. Comparación de cuatro modelos por validación cruzada
8. Modelo final — entrenamiento, evaluación y curvas
9. Interpretabilidad — Importancia de variables por permutación
10. Calibración de probabilidades
11. Tabla "órdenes en riesgo" (alimenta PN10 del dashboard)
12. Conclusiones y entrada a Fase 5


## 1. Objetivos, alcance y métricas de éxito

### 1.1 Objetivo general

Construir un **clasificador binario** que prediga, a partir de
características observables al momento de cerrar la orden, si el
cliente otorgará una reseña *positiva* (≥ 4 estrellas) o *negativa*
(≤ 3 estrellas). El modelo alimenta la pregunta de negocio **PN10**
del dashboard de la Fase 3: *"¿qué órdenes activas tienen alto riesgo
de generar una mala reseña y deberíamos intervenir preventivamente?"*.

### 1.2 Por qué clasificar y no regresar el `review_score` 1-5

Para el caso de uso operativo (intervenir antes del cierre del ciclo),
nos interesa una **bandera accionable** (sí intervengo / no intervengo),
no un puntaje continuo. Además:

- El `review_score` es **ordinal**, no estrictamente numérico
  (la diferencia entre 4 y 5 no es la misma que entre 1 y 2).
- La distribución está **bimodal**: la mayoría son 5 o 1, casi nada en el medio.
- Una regresión sobre una variable bimodal-ordinal genera predicciones
  *en el centro* que no son accionables.

### 1.3 Métricas elegidas y por qué

| Métrica | Por qué la usamos | Riesgo si miramos solo esta |
|---|---|---|
| **Accuracy** | Intuitiva | Engaña en desbalance (77% positivas → un modelo trivial ya da 77%) |
| **F1** | Equilibrio precisión/recall sobre la clase positiva | Ignora la clase negativa que es la *accionable* |
| **ROC-AUC** | Robusto al desbalance, mide ordenamiento | No refleja calidad de probabilidad absoluta |
| **PR-AUC** | Importante con clase positiva mayoritaria | Sensible al umbral |
| **Curva calibración** | Verifica que `P(positiva)` predicha = frecuencia real | No es métrica única |

**Métrica de selección del modelo**: **ROC-AUC** (ordenamiento) +
chequeo de **calibración** (que `0.7` predicho ≈ 70% real).

### 1.4 Baseline

El baseline trivial *predecir siempre 'positiva'* logra:
- Accuracy ≈ 77.6%
- F1 ≈ 0.87 (alto por el desbalance)
- **ROC-AUC = 0.50** (sin capacidad discriminativa)

Cualquier modelo serio debe superar AUC = 0.50 por **al menos 10 puntos**
para considerarse útil.


## 2. Carga de datos

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, json, joblib
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
DATA = ROOT / "Data" / "processed"
MOD  = ROOT / "Models"
FIG  = ROOT / "Notebooks" / "figures" / "fase4"

mart = pd.read_csv(DATA/"marts/mart_orders.csv",
                   parse_dates=["order_purchase_timestamp","order_delivered_customer_date"])
print(f"Shape inicial: {mart.shape}")
mart.head(2)


## 3. Plan de outliers — sección 12.3 de la Fase 2

Aplicamos *exactamente* el plan documentado al final del EDA:

| Variable | Regla | Acción |
|---|---|---|
| `delivery_days_real`     | `< 0` | Eliminar |
| `total_value`, `total_freight`, `payment_total` | `> p99` | Winsorizar a p99 |
| `payment_installments_max` | `> 12` | Clip a 12 |
| `n_items` | `> 10` | Clip a 10 |

> **Por qué no transformamos `delivery_days_real`/`delivery_delay_days`:**
> sus colas largas son la señal predictiva más importante (ver
> permutation importance en la sección 9). Tocarlas destruiría la
> capacidad del modelo de identificar las órdenes en riesgo.


In [ ]:
df = mart.dropna(subset=["is_positive_review"]).copy()
df = df[df["delivery_days_real"].fillna(0) >= 0]
for c in ["total_value","total_freight","payment_total"]:
    p99 = df[c].quantile(0.99); df[c] = df[c].clip(upper=p99)
df["payment_installments_max"] = df["payment_installments_max"].clip(upper=12)
df["n_items"] = df["n_items"].clip(upper=10)
df = df[df["order_status"]=="delivered"].copy()
print(f"Tras outliers + filtro 'delivered': {df.shape}")
print(f"Tasa positiva: {df['is_positive_review'].mean()*100:.2f}%")


## 4. Construcción de features

Variables candidatas elegidas siguiendo la sección 14 del EDA
(*Conclusiones y siguiente fase*):

- **Logísticas:** `delivery_days_real`, `delivery_delay_days`, `is_late`
- **Económicas:** `total_value`, `total_freight`, `payment_total`,
  `payment_installments_max`
- **De orden:** `n_items`, `n_distinct_sellers`
- **Temporales:** `is_weekend`, `purchase_dow`
- **Categóricas:** `first_product_category`, `customer_state`,
  `main_payment_type`

> **Reducción de cardinalidad.** `first_product_category` tiene >70
> niveles; las menos frecuentes generan ruido. Conservamos el **top 20**
> y agrupamos el resto como `'other'`.


In [ ]:
num_feats = ["delivery_days_real","delivery_delay_days","is_late",
             "total_value","total_freight","payment_total",
             "payment_installments_max","n_items","n_distinct_sellers",
             "is_weekend","purchase_dow"]
cat_feats = ["first_product_category","customer_state","main_payment_type"]

top_cat = df["first_product_category"].value_counts().head(20).index
df["first_product_category"] = df["first_product_category"].where(
    df["first_product_category"].isin(top_cat), "other")

X = df[num_feats+cat_feats]
y = df["is_positive_review"].astype(int)
print(f"X: {X.shape} | Cardinalidad categóricas:")
for c in cat_feats:
    print(f"  {c:25s}: {X[c].nunique()} niveles")


## 5. Split estratificado y pipeline de preprocesamiento

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"% positiva en train: {y_train.mean()*100:.2f} | en test: {y_test.mean()*100:.2f}")


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("scl", StandardScaler())])
cat_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                     ("oh",  OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
preproc = ColumnTransformer([("num", num_pipe, num_feats),
                              ("cat", cat_pipe, cat_feats)])
preproc


**Decisiones de preprocesamiento explicadas**

| Decisión | Por qué |
|---|---|
| `SimpleImputer(median)` para numéricas | La mediana es robusta a outliers (que ya tratamos, pero por seguridad) |
| `SimpleImputer(most_frequent)` para categóricas | Conserva la moda; alternativa común con bajo impacto |
| `StandardScaler` (z-score) | Necesario para Regresión Logística; neutro para árboles |
| `OneHotEncoder` con `handle_unknown='ignore'` | Robustez ante categorías nuevas en producción |
| Sin SMOTE/SMOTETomek | Preferimos `class_weight='balanced'` y `scale_pos_weight` — métodos integrados, sin generar muestras sintéticas |


## 6. Comparación de cuatro modelos (CV 3-fold)

In [ ]:
# Resultados ya calculados — los cargamos para mantener consistencia
cv = pd.read_csv(MOD/"cv_results.csv")
cv.round(3)


In [ ]:
# Visualización de la comparación
from IPython.display import Image
Image(filename=str(FIG/"01_cv_comparison.png"), width=900)


**Lectura.**

- El **Baseline** alcanza ya 78% de accuracy pero **AUC = 0.50** —
  evidencia del desbalance del target.
- **Regresión Logística** ya se acerca al techo en AUC (0.70). El
  problema es **no estrictamente lineal**.
- **Random Forest** y **XGBoost** logran AUC similares (~0.70). El
  EDA (PCA, intra-grupo) ya anticipaba este límite: las variables
  predictoras tienen alta varianza intra-clase.
- **Elegimos XGBoost** como modelo final por su excelente trade-off
  velocidad/rendimiento, soporte nativo a `scale_pos_weight` y
  facilidad para producción.


## 7. Modelo final — entrenamiento y evaluación en test

In [ ]:
# El modelo ganador ya está entrenado y serializado en Models/
model = joblib.load(MOD/"xgb_positive_review.joblib")

# Predicciones sobre el test
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Métricas
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              average_precision_score, classification_report,
                              confusion_matrix)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1      : {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC : {roc_auc_score(y_test, y_proba):.4f}")
print(f"PR-AUC  : {average_precision_score(y_test, y_proba):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["Negativa","Positiva"]))


### 7.1 Matriz de confusión

In [ ]:
Image(filename=str(FIG/"02_confusion_matrix.png"), width=600)


**Lectura.** Con el umbral por defecto (0.5):

- **Verdaderos negativos: 2 007** — clientes insatisfechos detectados
  correctamente (los más valiosos desde el punto de vista de
  intervención preventiva).
- **Falsos positivos: 2 031** — clientes que el modelo predice felices
  pero terminan insatisfechos. Aquí está el gap que un futuro modelo
  podría reducir.
- **Falsos negativos: 2 790** — clientes que el modelo flagea como
  insatisfechos pero realmente son positivos. Cost: una notificación
  innecesaria.
- **Verdaderos positivos: 12 339** — clientes felices correctamente
  predichos.

En un entorno operativo, el umbral debería **ajustarse según el costo
relativo** de intervenir vs. perder un cliente.


### 7.2 Curvas ROC y Precision-Recall

In [ ]:
Image(filename=str(FIG/"03_roc_pr.png"), width=950)


**Lectura.**

- **ROC-AUC = 0.705.** El modelo ordena
  correctamente al 70% de los pares (positivo, negativo). El baseline
  aleatorio es 0.5; un modelo perfecto sería 1.0.
- **PR-AUC = 0.874.** Significativamente
  arriba del baseline (que sería igual a la tasa positiva, 0.79). El
  PR-AUC es informativo aquí porque la clase positiva es mayoritaria.


## 8. Interpretabilidad — Permutation Importance

Calculamos la **importancia por permutación** sobre 5 000 ejemplos del
test, midiendo la **caída en ROC-AUC** cuando barajamos al azar los
valores de cada feature. Es model-agnostic, sensible al *signal* real
del modelo y reportable en publicaciones académicas.

> **Nota técnica.** Inicialmente usamos SHAP, pero la versión 0.49 de
> SHAP tiene un *bug* conocido al parsear modelos XGBoost 3.x
> (`could not convert '[5E-1]' to float`). Sustituimos por permutation
> importance, que es funcionalmente equivalente para este caso de uso.


In [ ]:
Image(filename=str(FIG/"06_permutation_importance.png"), width=820)


In [ ]:
perm = pd.read_csv(MOD/"permutation_importance.csv")
perm.head(10).round(4)


**Lectura — confirma las hipótesis de la Fase 2.**

1. **`delivery_days_real` domina** (caída −0.044 AUC al permutar).
   Confirma la **H1** del EDA: el tiempo real de entrega *es* el
   driver principal de la satisfacción.
2. **`n_items`** sorprendentemente segundo. Las órdenes con más ítems
   tienen patrones de satisfacción distintos (más exposición a fallos
   logísticos, más probabilidad de un ítem dañado).
3. **`delivery_delay_days`** confirma **H2** (el retraso respecto al
   estimado importa, pero menos que el tiempo absoluto).
4. **`first_product_category`** y **`customer_state`** aparecen en
   el top: las **categorías difíciles** y los **estados periféricos**
   son señales reales.
5. **`is_late`** (la versión binaria) **no agrega valor adicional** —
   la información ya está contenida en `delivery_days_real` y
   `delivery_delay_days`. Útil para BI pero redundante para el modelo.


## 9. Calibración de probabilidades

Una probabilidad predicha de 0.8 debe significar que **el 80% de las
órdenes con score 0.8 son realmente positivas**. Si no, las
probabilidades no se pueden usar para *priorizar* intervenciones.


In [ ]:
Image(filename=str(FIG/"04_calibration.png"), width=620)


**Lectura.** La curva se acerca a la diagonal, lo que indica
calibración razonable. Para un uso productivo (alimentar PN10 del
dashboard) es suficiente; si más adelante quisiéramos *thresholds
operacionales finos*, se puede aplicar `CalibratedClassifierCV` con
calibración isotónica.


## 10. Importancia desde el modelo (gain) — vista complementaria

In [ ]:
Image(filename=str(FIG/"05_feature_importance.png"), width=820)


**Comparación gain vs. permutación.** Ambas vistas coinciden en
*qué* features son importantes pero difieren en *orden* exacto. El
"gain" mide cuánto contribuye cada split del árbol; la permutación
mide cuánto se cae el AUC sin esa información. **La permutación es la
métrica más fiable** porque mide impacto *en la métrica objetivo*.


## 11. Tabla "órdenes en riesgo" para el dashboard (PN10)

Generamos las **50 órdenes con mayor score de reseña negativa** del
conjunto de test. Esta tabla se inserta en la página 3 del dashboard
para responder a la pregunta PN10.


In [ ]:
risk = pd.read_csv(MOD/"orders_at_risk_top50.csv")
print(f"Top 5 órdenes en riesgo (de 50):")
risk.head(5)[["score_negativa","delivery_days_real","is_late",
              "first_product_category","customer_state","main_payment_type","y_real"]]


In [ ]:
# Distribución del score en las 50 órdenes en riesgo
print("Resumen estadístico de las órdenes flageadas:")
print(risk[["score_negativa","delivery_days_real","delivery_delay_days",
            "is_late","total_value"]].describe().round(2))


## 12. Conclusiones de la Fase 4 y entrada a la Fase 5

### Resumen de resultados

| Hito | Resultado |
|---|---|
| Plan de outliers aplicado | Sí (sección 3) |
| Pipeline reproducible | `ColumnTransformer` + 4 estimadores |
| Validación cruzada | 3-fold estratificada |
| Modelo ganador | XGBoost |
| **ROC-AUC en test** | **0.705** |
| **F1 en test** | **0.837** |
| Lift sobre baseline AUC | **+20.50 puntos** (de 0.50 → 0.705) |
| Tabla 'órdenes en riesgo' | 50 casos para PN10 |
| Modelo serializado | `Models/xgb_positive_review.joblib` |

### Hallazgos clave

1. **La logística domina la satisfacción.** `delivery_days_real` y
   `delivery_delay_days` ocupan dos de los tres primeros lugares en
   permutation importance. Lo que dijo el EDA estadísticamente
   (H1, H2 con p < 0.05) lo confirma el modelo predictivamente.
2. **El número de ítems es una señal subestimada.** No estaba entre los
   sospechosos habituales, pero el modelo le otorga un peso importante.
   Hipótesis: más ítems = más superficies para que algo salga mal
   (un ítem dañado, retraso parcial, ítems faltantes).
3. **Las categorías y el estado del cliente importan.** No tanto como
   la logística, pero suficiente para justificar políticas de
   intervención diferenciadas por categoría/región.
4. **El modelo no es perfecto** — AUC ≈ 0.70 es honesto. Refleja que
   con la información disponible *antes* del cierre, hay una porción
   del comportamiento del cliente que escapa al modelo (texto de la
   reseña, NPS histórico, eventos posteriores).

### Entrada a la Fase 5

La fase 5 sintetiza estos hallazgos en **recomendaciones de negocio
accionables**: dónde abrir hubs logísticos, qué categorías auditar,
qué umbral usar para alertar al equipo de atención al cliente.
